# Marine Spatial Planning Violation Signals

Colab/Jupyter starter notebook using Google Earth Engine and `geemap`.

The outputs are review signals, not legal conclusions.

## 1. Install and import packages

In Colab, uncomment and run the install line first.

In [3]:
!pip install -q earthengine-api geemap geopandas pandas shapely

import datetime as dt

import ee
import geemap
import pandas as pd


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()


Successfully saved authorization token.


## 2. Analysis parameters

In [5]:
START_DATE = "2022-01-01"
END_DATE = "2024-01-01"
INTERVAL_MONTHS = 6  # use 3 or 6
CLOUD_PROBABILITY_THRESHOLD = 40
NDWI_THRESHOLD = 0.0

## 3. Draw or load the coverage area

Draw a polygon on the map using the left toolbar, then run the next cell to capture it as `aoi`. If nothing is drawn, the notebook uses the fallback polygon.

In [9]:
Map = geemap.Map(center=[-6.2, 106.8], zoom=10)
Map.add_basemap("SATELLITE")
Map

Map(center=[-6.2, 106.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [5]:
fallback_coords = [
    [106.70, -6.10],
    [106.90, -6.10],
    [106.90, -6.30],
    [106.70, -6.30],
    [106.70, -6.10],
]

aoi = Map.user_roi
if aoi is None:
    aoi = ee.Geometry.Polygon([fallback_coords])
    print("No drawn AOI found. Using fallback example polygon.")
else:
    print("Using drawn AOI from geemap.")

Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map.centerObject(aoi, 11)

Using drawn AOI from geemap.


## 4. Earth Engine helper functions

In [6]:
def date_windows(start_date, end_date, interval_months=3):
    start = dt.date.fromisoformat(start_date)
    end = dt.date.fromisoformat(end_date)
    windows = []
    current = start

    while current < end:
        month = current.month - 1 + interval_months
        year = current.year + month // 12
        month = month % 12 + 1
        day = min(current.day, 28)
        next_date = dt.date(year, month, day)
        if next_date > end:
            next_date = end
        windows.append((current.isoformat(), next_date.isoformat()))
        current = next_date

    return windows


def mask_s2_clouds(image):
    cloud_prob = ee.Image(image.get("cloud_mask")).select("probability")
    scl = image.select("SCL")

    clear_cloud_prob = cloud_prob.lt(CLOUD_PROBABILITY_THRESHOLD)
    clear_scl = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))

    return image.updateMask(clear_cloud_prob.And(clear_scl)).divide(10000).copyProperties(
        image, ["system:time_start"]
    )


def sentinel2_collection(aoi, start_date, end_date):
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(aoi).filterDate(start_date, end_date)
    clouds = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(aoi).filterDate(start_date, end_date)

    joined = ee.ImageCollection(
        ee.Join.saveFirst("cloud_mask").apply(
            primary=s2,
            secondary=clouds,
            condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
        )
    )

    return joined.map(mask_s2_clouds)


def add_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")
    ndbi = image.normalizedDifference(["B11", "B8"]).rename("NDBI")
    return image.addBands([ndvi, ndwi, ndbi])


def make_composite(aoi, start_date, end_date):
    collection = sentinel2_collection(aoi, start_date, end_date)
    return add_indices(collection.median().clip(aoi)).set(
        {"window_start": start_date, "window_end": end_date, "image_count": collection.size()}
    )

## 5. Build cloud-masked time series

In [7]:
windows = date_windows(START_DATE, END_DATE, INTERVAL_MONTHS)
composites = [make_composite(aoi, start, end) for start, end in windows]

summary_rows = []
for image, (start, end) in zip(composites, windows):
    summary_rows.append(
        {
            "window_start": start,
            "window_end": end,
            "sentinel2_image_count": image.get("image_count").getInfo(),
        }
    )

pd.DataFrame(summary_rows)

,window_start,window_end,sentinel2_image_count
0,2022-01-01,2022-07-01,70
1,2022-07-01,2023-01-01,74
2,2023-01-01,2023-07-01,72
3,2023-07-01,2024-01-01,72


## 6. Map index changes

In [8]:
start_image = composites[0]
end_image = composites[-1]

ndvi_change = end_image.select("NDVI").subtract(start_image.select("NDVI")).rename("NDVI_change")
ndwi_change = end_image.select("NDWI").subtract(start_image.select("NDWI")).rename("NDWI_change")
ndbi_change = end_image.select("NDBI").subtract(start_image.select("NDBI")).rename("NDBI_change")

change_map = geemap.Map()
change_map.centerObject(aoi, 11)
change_map.add_basemap("SATELLITE")
change_map.addLayer(aoi, {"color": "yellow"}, "AOI")
change_map.addLayer(start_image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "Start true color")
change_map.addLayer(end_image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "End true color")
change_map.addLayer(ndvi_change, {"min": -0.5, "max": 0.5, "palette": ["brown", "white", "green"]}, "NDVI change")
change_map.addLayer(ndwi_change, {"min": -0.5, "max": 0.5, "palette": ["red", "white", "blue"]}, "NDWI change")
change_map.addLayer(ndbi_change, {"min": -0.5, "max": 0.5, "palette": ["blue", "white", "red"]}, "NDBI change")
change_map

Map(center=[-6.075963209919964, 106.84660000000062], controls=(WidgetControl(options=['position', 'transparent…

## 7. Land/water change from NDWI

In [9]:
start_water = start_image.select("NDWI").gt(NDWI_THRESHOLD)
end_water = end_image.select("NDWI").gt(NDWI_THRESHOLD)

water_to_land = start_water.And(end_water.Not()).rename("water_to_land")
land_to_water = start_water.Not().And(end_water).rename("land_to_water")

pixel_area_ha = ee.Image.pixelArea().divide(10000)
area_image = ee.Image.cat(
    [
        water_to_land.multiply(pixel_area_ha).rename("water_to_land_ha"),
        land_to_water.multiply(pixel_area_ha).rename("land_to_water_ha"),
    ]
)

area_stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=10, maxPixels=1e13
).getInfo()

pd.DataFrame([area_stats])

,land_to_water_ha,water_to_land_ha
0,552.42946,1437.852877


In [10]:
water_map = geemap.Map()
water_map.centerObject(aoi, 11)
water_map.add_basemap("SATELLITE")
water_map.addLayer(water_to_land.selfMask(), {"palette": ["red"]}, "Water to land")
water_map.addLayer(land_to_water.selfMask(), {"palette": ["cyan"]}, "Land to water")
water_map.addLayer(aoi, {"color": "yellow"}, "AOI")
water_map

Map(center=[-6.075963209919964, 106.84660000000062], controls=(WidgetControl(options=['position', 'transparent…

## 8. Coral reef and business entity placeholders

Allen Coral Atlas can be added after confirming the correct Earth Engine asset ID for the study region.

Business entities are not available directly from Earth Engine. Likely sources include official licensing/company registries, local permit datasets, OpenStreetMap/Overpass, ports, industrial estates, concessions, or aquaculture datasets.

In [ ]:
# Example after confirming an Allen Coral Atlas asset:
# coral = ee.FeatureCollection("projects/.../allen_coral_atlas_asset").filterBounds(aoi)
# change_map.addLayer(coral, {"color": "magenta"}, "Allen Coral Atlas")